# 01 — Daily incremental pipeline
This notebook calls the maintained Python pipeline in `src/finops_cloud/pipelines/daily_incremental.py`.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
SOURCE_URI = ""
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    dbutils.widgets.text("source_uri", SOURCE_URI)
    ENVIRONMENT = dbutils.widgets.get("environment")
    SOURCE_URI = dbutils.widgets.get("source_uri")
except NameError:
    pass
if not SOURCE_URI:
    raise ValueError("Set source_uri to the daily Parquet file to load.")

In [ ]:
from finops_cloud.pipelines.daily_incremental import run

result = run(ENVIRONMENT, SOURCE_URI)
display(result)

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.runtime import get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
runs = spark_session.table(config.table("pipeline_run", "ops"))
display(runs.where(f"environment = '{ENVIRONMENT}'").orderBy("started_at", ascending=False).limit(20))